In [ ]:
from google.colab.output import eval_js
from base64 import b64decode
import numpy as np
import cv2, time, shutil
from pathlib import Path
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

# ---------------- CONFIG ----------------
CLASSES = ["Food","Water","Help","Fire","Sleep","Medicine"]
IMAGES_PER_CLASS = 200
OUT_DIR = Path("dataset_sign6_webcam")
OUT_DIR.mkdir(exist_ok=True)

BN_HINT = {"Food":"খাবার","Water":"পানি","Help":"সাহায্য","Fire":"আগুন","Sleep":"ঘুম","Medicine":"ওষুধ"}

# Quality filters (tune if too strict)
BLUR_TH = 60.0
DIFF_TH = 3.0

# Capture pacing
SAVE_FPS = 6.0
save_interval = 1.0 / SAVE_FPS

# Visual hints / UI
COUNTDOWN_SECONDS = 5
BREAK_BETWEEN_CLASSES_SEC = 120   # 2 minutes
SHOW_PREVIEW_EVERY = 25          # show a preview image every N saves

# ROI settings
ROI_SCALE = 0.65
OUT_SIZE = (96, 96)

# ---------------- Webcam JS ----------------
def js_capture():
    js = r"""
    async function capture() {
      const div = document.createElement('div');
      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});
      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      const ctx = canvas.getContext('2d');

      return new Promise((resolve) => {
        setTimeout(async () => {
          ctx.drawImage(video, 0, 0);
          stream.getTracks().forEach(track => track.stop());
          div.remove();
          resolve(canvas.toDataURL('image/jpeg', 0.8));
        }, 150);
      });
    }
    capture();
    """
    return eval_js(js)

# ---------------- Helpers ----------------
def center_square(gray, scale=0.65, out_size=(96,96)):
    h, w = gray.shape
    s = int(min(h,w)*scale)
    y0 = (h - s)//2
    x0 = (w - s)//2
    roi = gray[y0:y0+s, x0:x0+s]
    roi = cv2.resize(roi, out_size, interpolation=cv2.INTER_AREA)
    return roi

def blur_score(img):
    return float(cv2.Laplacian(img, cv2.CV_64F).var())

def mean_abs_diff(a,b):
    return float(np.mean(cv2.absdiff(a,b)))

def show_preview(full_bgr, roi_gray, cls, saved, b, d):
    clear_output(wait=True)

    # Draw ROI rectangle on full image for visualization
    vis = full_bgr.copy()
    h, w = vis.shape[:2]
    s = int(min(h,w)*ROI_SCALE)
    y0 = (h - s)//2
    x0 = (w - s)//2
    cv2.rectangle(vis, (x0, y0), (x0+s, y0+s), (0,255,0), 3)

    # Convert BGR->RGB for matplotlib
    vis_rgb = cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    plt.title(f"Live frame (ROI box) | Class: {cls}")
    plt.imshow(vis_rgb)
    plt.axis("off")

    plt.subplot(1,2,2)
    plt.title(f"Saved ROI (96x96) | saved={saved}\nblur={b:.1f} diff={d:.1f}")
    plt.imshow(roi_gray, cmap="gray")
    plt.axis("off")
    plt.show()

# ---------------- Main capture loop ----------------
for ci, cls in enumerate(CLASSES):
    (OUT_DIR/cls).mkdir(exist_ok=True)

    print(f"\nGet ready for: {cls} ({BN_HINT[cls]})")
    for t in range(COUNTDOWN_SECONDS, 0, -1):
        print("Starting in", t)
        time.sleep(1)

    saved = 0
    last_img = None
    last_t = 0.0
    last_preview_time = 0.0

    while saved < IMAGES_PER_CLASS:
        data_url = js_capture()
        jpg = b64decode(data_url.split(',')[1])
        arr = np.frombuffer(jpg, dtype=np.uint8)
        frame_bgr = cv2.imdecode(arr, cv2.IMREAD_COLOR)

        gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
        roi = center_square(gray, scale=ROI_SCALE, out_size=OUT_SIZE)

        b = blur_score(roi)
        d = 999.0 if last_img is None else mean_abs_diff(roi, last_img)

        now = time.time()
        if (now-last_t) >= save_interval and b >= BLUR_TH and (last_img is None or d >= DIFF_TH):
            out = OUT_DIR/cls/f"{cls.lower()}_{saved:06d}.png"
            cv2.imwrite(str(out), roi)
            last_img = roi
            last_t = now
            saved += 1

            # Print hint line so you know it saved something
            if saved % 10 == 0:
                print(f"{cls}: saved {saved}/{IMAGES_PER_CLASS} | blur={b:.1f} diff={d:.1f}")

            # Show preview occasionally so you can SEE what is being captured
            if saved % SHOW_PREVIEW_EVERY == 0:
                show_preview(frame_bgr, roi, cls, saved, b, d)

    print(f"\n[DONE] {cls}: saved {saved}")

    # 2-minute break between classes (except after last)
    if ci < len(CLASSES) - 1:
        for t in range(BREAK_BETWEEN_CLASSES_SEC, 0, -1):
            clear_output(wait=True)
            print(f"Break time: next class in {t} seconds")
            time.sleep(1)

print("Done. Zipping...")
shutil.make_archive("dataset_sign6_webcam", "zip", str(OUT_DIR))
print("Created dataset_sign6_webcam.zip")



Get ready for: Food (খাবার)
Starting in 5
Starting in 4
Starting in 3
Starting in 2
Starting in 1


MessageError: NotAllowedError: Permission denied

In [ ]:
shutil.make_archive("dataset_sign6_webcam", "zip", str(OUT_DIR))
print("Created dataset_sign6_webcam.zip")


Created dataset_sign6_webcam.zip


In [ ]:
# ============================================================
# Kaggle ONE-CELL training + INT8 export (FULL FIXED VERSION)
# - Works with TF/Keras builds that DON'T support label_smoothing
# - Uses MobileNetV2(alpha=0.35) transfer learning
# - Dataset format:
#   /kaggle/input/sign-6-words/Food/*.png ... etc
# - Outputs:
#   /kaggle/working/sign6_int8.tflite
#   /kaggle/working/sign6_int8.h
#   /kaggle/working/labels_bn.json
# ============================================================

import os, random, shutil, json
from pathlib import Path
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# (Usually available on Kaggle; if not, it will still train but skip report)
try:
    from sklearn.metrics import confusion_matrix, classification_report
    SKLEARN_OK = True
except Exception:
    SKLEARN_OK = False

# ------------------------
# Reproducibility
# ------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ------------------------
# Paths
# ------------------------
DATASET_ROOT = Path("/content/dataset_sign6_webcam")   # <-- your dataset root
WORK_ROOT = Path("/content/outputs")

SPLIT_DIR = WORK_ROOT / "sign6_split"
TRAIN_DIR = SPLIT_DIR / "train"
VAL_DIR   = SPLIT_DIR / "val"
TEST_DIR  = SPLIT_DIR / "test"

TFLITE_PATH = WORK_ROOT / "sign6_int8.tflite"
HEADER_PATH = WORK_ROOT / "sign6_int8.h"
LABELS_JSON = WORK_ROOT / "labels_bn.json"
SAVEDMODEL_DIR = WORK_ROOT / "saved_sign6"

# ------------------------
# Classes + Bengali mapping
# ------------------------
CLASSES = ["Food","Water","Help","Fire","Sleep","Medicine"]
BN_WORD = {"Food":"খাবার","Water":"পানি","Help":"সাহায্য","Fire":"আগুন","Sleep":"ঘুম","Medicine":"ওষুধ"}
BN_MSG  = {"Food":"আমার খাবার দরকার","Water":"আমার পানি দরকার","Help":"সাহায্য করুন","Fire":"আগুন! বিপদ","Sleep":"আমার ঘুম দরকার","Medicine":"আমার ওষুধ দরকার"}

NUM_CLASSES = len(CLASSES)

# ------------------------
# Training params
# ------------------------
IMG_SIZE = (96, 96)
BATCH = 32
EPOCHS_HEAD = 15
EPOCHS_FT = 64

# ============================================================
# 1) Validate dataset structure
# ============================================================
if not DATASET_ROOT.exists():
    raise RuntimeError(f"DATASET_ROOT not found: {DATASET_ROOT}")

missing = [c for c in CLASSES if not (DATASET_ROOT / c).exists()]
if missing:
    raise RuntimeError(
        f"Missing class folders: {missing}\n"
        f"Expected: {DATASET_ROOT}/Fire/*.png etc."
    )

def list_images(folder: Path):
    exts = (".png", ".jpg", ".jpeg", ".bmp", ".webp")
    return sorted([p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in exts])

counts = {c: len(list_images(DATASET_ROOT / c)) for c in CLASSES}
print("Images per class:", counts)

# ============================================================
# 2) Split 80/10/10 into /kaggle/working/sign6_split
# ============================================================
def mkdirs():
    for sp in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
        for c in CLASSES:
            (sp / c).mkdir(parents=True, exist_ok=True)

def split_ready():
    return all((TRAIN_DIR/c).exists() for c in CLASSES) and any((TRAIN_DIR/CLASSES[0]).glob("*"))

if SPLIT_DIR.exists() and not split_ready():
    shutil.rmtree(SPLIT_DIR)

if not split_ready():
    if SPLIT_DIR.exists():
        shutil.rmtree(SPLIT_DIR)
    mkdirs()

    rng = random.Random(SEED)
    for c in CLASSES:
        files = list_images(DATASET_ROOT / c)
        rng.shuffle(files)
        n = len(files)
        n_train = int(0.80 * n)
        n_val   = int(0.10 * n)

        train_files = files[:n_train]
        val_files   = files[n_train:n_train+n_val]
        test_files  = files[n_train+n_val:]

        for f in train_files: shutil.copy2(str(f), str(TRAIN_DIR/c/f.name))
        for f in val_files:   shutil.copy2(str(f), str(VAL_DIR/c/f.name))
        for f in test_files:  shutil.copy2(str(f), str(TEST_DIR/c/f.name))

    print("Created split at:", SPLIT_DIR)
else:
    print("Using existing split at:", SPLIT_DIR)

# ============================================================
# 3) Datasets (RGB for ImageNet backbone)
# ============================================================
def make_ds(dirpath, shuffle):
    return tf.keras.utils.image_dataset_from_directory(
        dirpath,
        labels="inferred",
        label_mode="int",
        class_names=CLASSES,
        color_mode="rgb",
        image_size=IMG_SIZE,
        batch_size=BATCH,
        shuffle=shuffle,
        seed=SEED,
    )

train_ds_raw = make_ds(str(TRAIN_DIR), shuffle=True)
val_ds       = make_ds(str(VAL_DIR), shuffle=False)
test_ds      = make_ds(str(TEST_DIR), shuffle=False)

AUTOTUNE = tf.data.AUTOTUNE

# Augment OUTSIDE model (keeps export clean)
def augment(x, y):
    x = tf.cast(x, tf.float32)
    x = tf.image.random_flip_left_right(x)
    x = tf.image.random_brightness(x, 0.10)
    x = tf.image.random_contrast(x, 0.85, 1.15)
    return x, y

train_ds = train_ds_raw.map(augment, num_parallel_calls=AUTOTUNE).cache().prefetch(AUTOTUNE)
val_ds   = val_ds.cache().prefetch(AUTOTUNE)
test_ds  = test_ds.cache().prefetch(AUTOTUNE)

# ============================================================
# 4) Custom sparse CE with label smoothing (works everywhere)
# ============================================================
def make_sparse_ce_with_smoothing(num_classes, smoothing):
    smoothing = float(smoothing)
    def loss_fn(y_true, y_pred):
        y_true = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        y_oh = tf.one_hot(y_true, num_classes)  # (B, C)
        y_sm = y_oh * (1.0 - smoothing) + (smoothing / num_classes)
        return tf.keras.losses.categorical_crossentropy(y_sm, y_pred)
    return loss_fn

# ============================================================
# 5) Model: MobileNetV2(alpha=0.35) transfer learning
# ============================================================
# Preprocess inside the model: MobileNetV2 expects [-1, 1]
preprocess = keras.Sequential([layers.Rescaling(1./127.5, offset=-1.0)], name="preprocess")

base = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3),
    include_top=False,
    weights="imagenet",
    alpha=0.35
)
base.trainable = False  # Stage 1

inputs = keras.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
x = preprocess(inputs)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.35)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
model = keras.Model(inputs, outputs, name="Sign6_MNetV2_035")
model.summary()

callbacks = [
    keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(patience=3, factor=0.5, min_lr=1e-6),
]

print("\n--- Stage 1: train head ---")
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=make_sparse_ce_with_smoothing(NUM_CLASSES, smoothing=0.05),
    metrics=["accuracy"]
)
model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD, callbacks=callbacks)

print("\n--- Stage 2: fine-tune top layers ---")
base.trainable = True

# Keep BN frozen for stability on small datasets
for layer in base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

# Unfreeze only last ~30 layers
for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(1e-4),
    loss=make_sparse_ce_with_smoothing(NUM_CLASSES, smoothing=0.02),
    metrics=["accuracy"]
)
model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FT, callbacks=callbacks)

# ============================================================
# 6) Evaluate + sanity checks
# ============================================================
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print("\nTest accuracy:", float(test_acc))

y_true, y_pred = [], []
for xb, yb in test_ds:
    probs = model.predict(xb, verbose=0)
    y_true.extend(yb.numpy().tolist())
    y_pred.extend(np.argmax(probs, axis=1).tolist())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print("\nPrediction distribution (should not be stuck):")
print({CLASSES[i]: int(np.sum(y_pred == i)) for i in range(NUM_CLASSES)})

if SKLEARN_OK:
    cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    print("\nConfusion matrix (rows=true, cols=pred):\n", cm)
    print("\nPer-class report:\n", classification_report(y_true, y_pred, target_names=CLASSES, digits=4))

# ============================================================
# 7) Save Bengali labels JSON
# ============================================================
with open(LABELS_JSON, "w", encoding="utf-8") as f:
    json.dump(
        {"classes": CLASSES,
         "bn_word": [BN_WORD[c] for c in CLASSES],
         "bn_msg":  [BN_MSG[c] for c in CLASSES]},
        f, ensure_ascii=False, indent=2
    )
print("Saved:", str(LABELS_JSON))

# ============================================================
# 8) Export SavedModel + INT8 TFLite + C header
# ============================================================
if SAVEDMODEL_DIR.exists():
    shutil.rmtree(SAVEDMODEL_DIR)

try:
    model.export(str(SAVEDMODEL_DIR))   # Keras 3
except Exception:
    tf.saved_model.save(model, str(SAVEDMODEL_DIR))

def representative_data_gen():
    # Use non-augmented raw train set for calibration
    for images, _ in train_ds_raw.unbatch().batch(1).take(200):
        yield [tf.cast(images, tf.float32)]  # model preprocess inside

converter = tf.lite.TFLiteConverter.from_saved_model(str(SAVEDMODEL_DIR))
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_model = converter.convert()
TFLITE_PATH.write_bytes(tflite_model)
print("Saved:", str(TFLITE_PATH), "bytes:", len(tflite_model))

with open(HEADER_PATH, "w") as f:
    f.write("#pragma once\n#include <cstdint>\n\n")
    f.write("alignas(16) const unsigned char sign6_int8[] = {\n")
    for i, b in enumerate(tflite_model):
        if i % 12 == 0:
            f.write("  ")
        f.write(f"0x{b:02x}, ")
        if i % 12 == 11:
            f.write("\n")
    f.write("\n};\n")
    f.write(f"const unsigned int sign6_int8_len = {len(tflite_model)};\n")
print("Saved:", str(HEADER_PATH))

# Quant params for ESP32 input scaling
interpreter = tf.lite.Interpreter(model_path=str(TFLITE_PATH))
interpreter.allocate_tensors()
inp = interpreter.get_input_details()[0]
out = interpreter.get_output_details()[0]
print("\nTFLite input:", inp["shape"], inp["dtype"], "quant:", inp["quantization"])
print("TFLite output:", out["shape"], out["dtype"], "quant:", out["quantization"])


Images per class: {'Food': 200, 'Water': 200, 'Help': 200, 'Fire': 200, 'Sleep': 200, 'Medicine': 200}
Created split at: /content/outputs/sign6_split
Found 960 files belonging to 6 classes.
Found 120 files belonging to 6 classes.
Found 120 files belonging to 6 classes.
2019640/2019640 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "Sign6_MNetV2_035"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ preprocess (Sequential)         │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_0.35_96             │ (None, 3, 3, 1280)     │       410,208 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 6)              │         7,686 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 417,894 (1.59 MB)

 Trainable params: 7,686 (30.02 KB)

 Non-trainable params: 410,208 (1.56 MB)


--- Stage 1: train head ---
Epoch 1/15
30/30 ━━━━━━━━━━━━━━━━━━━━ 30s 490ms/step - accuracy: 0.5552 - loss: 1.2982 - val_accuracy: 0.9500 - val_loss: 0.5032 - learning_rate: 0.0010
Epoch 2/15
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9208 - loss: 0.4818 - val_accuracy: 0.9583 - val_loss: 0.3869 - learning_rate: 0.0010
Epoch 3/15
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.9625 - loss: 0.4049 - val_accuracy: 0.9750 - val_loss: 0.3576 - learning_rate: 0.0010
Epoch 4/15
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9740 - loss: 0.3619 - val_accuracy: 0.9750 - val_loss: 0.3410 - learning_rate: 0.0010
Epoch 5/15
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9792 - loss: 0.3499 - val_accuracy: 0.9750 - val_loss: 0.3292 - learning_rate: 0.0010
Epoch 6/15
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9854 - loss: 0.3309 - val_accuracy: 0.9833 - val_loss: 0.3221 - learning_rate: 0.0010
Epoch 7/15
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [ ]:
!zip -r outputs.zip /content/outputs

  adding: content/outputs/ (stored 0%)
  adding: content/outputs/sign6_int8.tflite (deflated 28%)
  adding: content/outputs/saved_sign6/ (stored 0%)
  adding: content/outputs/saved_sign6/variables/ (stored 0%)
  adding: content/outputs/saved_sign6/variables/variables.index (deflated 78%)
  adding: content/outputs/saved_sign6/variables/variables.data-00000-of-00001 (deflated 8%)
  adding: content/outputs/saved_sign6/fingerprint.pb (stored 0%)
  adding: content/outputs/saved_sign6/assets/ (stored 0%)
  adding: content/outputs/saved_sign6/saved_model.pb (deflated 91%)
  adding: content/outputs/sign6_int8.h (deflated 81%)
  adding: content/outputs/labels_bn.json (deflated 61%)
  adding: content/outputs/sign6_split/ (stored 0%)
  adding: content/outputs/sign6_split/val/ (stored 0%)
  adding: content/outputs/sign6_split/val/Fire/ (stored 0%)
  adding: content/outputs/sign6_split/val/Fire/fire_000060.png (stored 0%)
  adding: content/outputs/sign6_split/val/Fire/fire_000061.png (stored 0%)
  

In [ ]:
# ============================================================
# Colab: Smooth "real-time" webcam inference (no per-frame popups)
# - Keeps camera stream ON (no start/stop each frame)
# - Updates only prediction text smoothly
# Stop: Runtime -> Interrupt execution
# ============================================================

from google.colab.output import eval_js
from IPython.display import HTML, display
from base64 import b64decode
import numpy as np
import cv2, json, time, os
import tensorflow as tf

# ---- your paths ----
TFLITE_PATH = "/content/outputs/sign6_int8.tflite"
LABELS_JSON = "/content/outputs/labels_bn.json"

assert os.path.exists(TFLITE_PATH), "Model not found."
assert os.path.exists(LABELS_JSON), "labels_bn.json not found."

# -------- Load labels --------
with open(LABELS_JSON, "r", encoding="utf-8") as f:
    meta = json.load(f)
CLASSES = meta["classes"]
BN_WORD = meta["bn_word"]
BN_MSG  = meta["bn_msg"]
NUM_CLASSES = len(CLASSES)

# -------- Load TFLite --------
interpreter = tf.lite.Interpreter(model_path=TFLITE_PATH)
interpreter.allocate_tensors()
inp = interpreter.get_input_details()[0]
out = interpreter.get_output_details()[0]

in_h, in_w = int(inp["shape"][1]), int(inp["shape"][2])
in_scale, in_zero = inp["quantization"]
out_scale, out_zero = out["quantization"]

print("Input:", inp["shape"], inp["dtype"], "quant:", inp["quantization"])
print("Output:", out["shape"], out["dtype"], "quant:", out["quantization"])
print("Classes:", CLASSES)

# ============================================================
# 1) Start webcam ONCE + define captureFrame() in JS
# ============================================================
display(HTML("""
<div style="display:flex; gap:16px; align-items:flex-start;">
  <div>
    <div style="font-size:16px; font-weight:600; margin-bottom:6px;">Live camera</div>
    <video id="webcam" autoplay playsinline style="width:420px; border:1px solid #ddd; border-radius:10px;"></video>
    <canvas id="canvas" style="display:none;"></canvas>
  </div>
  <div id="predBox" style="min-width:380px; padding:12px; border:1px solid #ddd; border-radius:10px; font-family:Arial;">
    <div style="font-size:16px; font-weight:700;">Prediction</div>
    <div style="margin-top:8px; font-size:14px;">Waiting for frames...</div>
  </div>
</div>
"""))

eval_js(r"""
async function startWebcam() {
  if (window._streamStarted) return "ok";

  const video = document.getElementById('webcam');
  const stream = await navigator.mediaDevices.getUserMedia({video: {facingMode: "user"}});
  video.srcObject = stream;
  await video.play();

  window._streamStarted = true;

  // captureFrame reads from the SAME video every time (no stopping stream)
  window.captureFrame = function() {
    const canvas = document.getElementById('canvas');
    const ctx = canvas.getContext('2d');

    canvas.width = video.videoWidth;
    canvas.height = video.videoHeight;

    ctx.drawImage(video, 0, 0, canvas.width, canvas.height);
    return canvas.toDataURL('image/jpeg', 0.85);
  };

  return "ok";
}
startWebcam();
""")

# ============================================================
# 2) Preprocess + inference
# ============================================================
def preprocess_for_model(frame_bgr):
    # center crop square
    h, w = frame_bgr.shape[:2]
    s = min(h, w)
    y0 = (h - s) // 2
    x0 = (w - s) // 2
    crop = frame_bgr[y0:y0+s, x0:x0+s]

    crop = cv2.resize(crop, (in_w, in_h), interpolation=cv2.INTER_AREA)
    rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB).astype(np.float32)  # 0..255

    # Quantize float input to int8 using model's input quant params
    q = np.round(rgb / in_scale + in_zero)
    q = np.clip(q, -128, 127).astype(np.int8)
    q = np.expand_dims(q, axis=0)  # (1,H,W,3)
    return q

def run_inference_int8(x_int8):
    interpreter.set_tensor(inp["index"], x_int8)
    interpreter.invoke()
    out_int8 = interpreter.get_tensor(out["index"])[0].astype(np.int8)

    # Dequantize
    out_f = (out_int8.astype(np.float32) - out_zero) * out_scale

    # Normalize safely (in case quantized softmax is slightly off)
    out_f = np.maximum(out_f, 0.0)
    s = float(out_f.sum())
    if s > 0:
        out_f /= s
    return out_f

# ============================================================
# 3) Smooth + stability logic (no fast flicker)
# ============================================================
ema = np.ones((NUM_CLASSES,), dtype=np.float32) / NUM_CLASSES
EMA_ALPHA = 0.85          # closer to 1 => smoother (slower changes)
CONF_TH = 0.55            # minimum confidence
STABLE_FRAMES = 6         # require same prediction for N frames before "locking"
FPS_SLEEP = 0.06          # ~16 fps capture attempts (Colab speed varies)

stable_count = 0
last_idx = None
locked_idx = None

# Display handle (update only this, not the whole output)
pred_handle = display(HTML("<div></div>"), display_id=True)

print("\nRunning... (Interrupt execution to stop)\n")

while True:
    # Grab frame from already-running webcam
    data_url = eval_js("captureFrame()")
    jpg = b64decode(data_url.split(",")[1])
    arr = np.frombuffer(jpg, dtype=np.uint8)
    frame = cv2.imdecode(arr, cv2.IMREAD_COLOR)

    x = preprocess_for_model(frame)
    probs = run_inference_int8(x)

    # EMA smoothing
    ema = EMA_ALPHA * ema + (1.0 - EMA_ALPHA) * probs
    idx = int(np.argmax(ema))
    conf = float(ema[idx])

    # Stability counter
    if last_idx is None or idx != last_idx:
        stable_count = 1
        last_idx = idx
    else:
        stable_count += 1

    # Lock only if stable for enough frames and confident
    if stable_count >= STABLE_FRAMES and conf >= CONF_TH:
        locked_idx = idx

    # Build UI text
    top3 = np.argsort(-ema)[:3]
    if locked_idx is None:
        bn_line = "বাংলা বাক্য: (কম বিশ্বাসযোগ্য) আরও পরিষ্কার করে দেখান"
        word_line = "বাংলা শব্দ: -"
        lock_line = "Locked: -"
    else:
        word_line = f"বাংলা শব্দ: {BN_WORD[locked_idx]}"
        bn_line = f"বাংলা বাক্য: {BN_MSG[locked_idx]}"
        lock_line = f"Locked: {CLASSES[locked_idx]}"

    lines = []
    lines.append(f"<div style='font-size:16px; font-weight:800;'>Prediction (smooth)</div>")
    lines.append(f"<div style='margin-top:6px; font-size:14px;'>Current (EMA): <b>{CLASSES[idx]}</b> &nbsp; | conf={conf:.2f} &nbsp; | stable={stable_count}/{STABLE_FRAMES}</div>")
    lines.append(f"<div style='margin-top:6px; font-size:14px;'><b>{lock_line}</b></div>")
    lines.append(f"<div style='margin-top:10px; font-size:15px;'>{word_line}</div>")
    lines.append(f"<div style='margin-top:6px; font-size:15px;'>{bn_line}</div>")
    lines.append("<div style='margin-top:10px; font-size:13px;'>Top-3:</div>")
    lines.append("<div style='font-size:13px; line-height:1.5;'>"
                 + "<br>".join([f"{CLASSES[i]}: {float(ema[i]):.2f}" for i in top3])
                 + "</div>")

    pred_handle.update(HTML("<div style='padding:10px; border:1px solid #ddd; border-radius:10px; width:420px; font-family:Arial;'>"
                           + "".join(lines) + "</div>"))

    time.sleep(FPS_SLEEP)


Input: [ 1 96 96  3] <class 'numpy.int8'> quant: (1.0, -128)
Output: [1 6] <class 'numpy.int8'> quant: (0.00390625, -128)
Classes: ['Food', 'Water', 'Help', 'Fire', 'Sleep', 'Medicine']



Running... (Interrupt execution to stop)



KeyboardInterrupt: 

In [ ]:
# ============================================================
# Colab: Immediate Bengali Update Real-time Inference
# ============================================================

from google.colab.output import eval_js
from IPython.display import HTML, display
from base64 import b64decode
import numpy as np
import cv2, json, time, os
import tensorflow as tf

# ---- your paths ----
TFLITE_PATH = "/content/outputs/sign6_int8.tflite"
LABELS_JSON = "/content/outputs/labels_bn.json"

assert os.path.exists(TFLITE_PATH), "Model not found."
assert os.path.exists(LABELS_JSON), "labels_bn.json not found."

# -------- Load labels --------
with open(LABELS_JSON, "r", encoding="utf-8") as f:
    meta = json.load(f)
CLASSES = meta["classes"]
BN_WORD = meta["bn_word"]
BN_MSG  = meta["bn_msg"]
NUM_CLASSES = len(CLASSES)

# -------- Load TFLite --------
interpreter = tf.lite.Interpreter(model_path=TFLITE_PATH)
interpreter.allocate_tensors()
inp = interpreter.get_input_details()[0]
out = interpreter.get_output_details()[0]

in_h, in_w = int(inp["shape"][1]), int(inp["shape"][2])
in_scale, in_zero = inp["quantization"]
out_scale, out_zero = out["quantization"]

# ============================================================
# 1) Start webcam ONCE + JS helper
# ============================================================
display(HTML("""
<div style="display:flex; gap:16px; align-items:flex-start; margin-bottom: 20px;">
  <div>
    <div style="font-size:16px; font-weight:600; margin-bottom:6px;">Live camera</div>
    <video id="webcam" autoplay playsinline style="width:420px; border:1px solid #ddd; border-radius:10px;"></video>
    <canvas id="canvas" style="display:none;"></canvas>
  </div>
  <div id="ui-container"></div>
</div>
"""))

eval_js(r"""
async function startWebcam() {
  if (window._streamStarted) return "ok";
  const video = document.getElementById('webcam');
  const stream = await navigator.mediaDevices.getUserMedia({video: {facingMode: "user"}});
  video.srcObject = stream;
  await video.play();
  window._streamStarted = true;
  window.captureFrame = function() {
    const canvas = document.getElementById('canvas');
    const ctx = canvas.getContext('2d');
    canvas.width = video.videoWidth;
    canvas.height = video.videoHeight;
    ctx.drawImage(video, 0, 0, canvas.width, canvas.height);
    return canvas.toDataURL('image/jpeg', 0.85);
  };
  return "ok";
}
startWebcam();
""")

# ============================================================
# 2) Preprocess + inference functions
# ============================================================
def preprocess_for_model(frame_bgr):
    h, w = frame_bgr.shape[:2]
    s = min(h, w)
    y0, x0 = (h - s) // 2, (w - s) // 2
    crop = frame_bgr[y0:y0+s, x0:x0+s]
    crop = cv2.resize(crop, (in_w, in_h), interpolation=cv2.INTER_AREA)
    rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB).astype(np.float32)
    q = np.round(rgb / in_scale + in_zero)
    q = np.clip(q, -128, 127).astype(np.int8)
    return np.expand_dims(q, axis=0)

def run_inference_int8(x_int8):
    interpreter.set_tensor(inp["index"], x_int8)
    interpreter.invoke()
    out_int8 = interpreter.get_tensor(out["index"])[0].astype(np.int8)
    out_f = (out_int8.astype(np.float32) - out_zero) * out_scale
    out_f = np.maximum(out_f, 0.0)
    s = float(out_f.sum())
    if s > 0: out_f /= s
    return out_f

# ============================================================
# 3) Loop with Immediate Updates
# ============================================================
ema = np.ones((NUM_CLASSES,), dtype=np.float32) / NUM_CLASSES
EMA_ALPHA = 0.7  # Lowered slightly for faster response
CONF_TH = 0.45   # Threshold for immediate display

stable_count = 0
last_idx = None
pred_handle = display(HTML(""), display_id=True)

print("Running... (Interrupt to stop)")

try:
    while True:
        data_url = eval_js("captureFrame()")
        jpg = b64decode(data_url.split(",")[1])
        arr = np.frombuffer(jpg, dtype=np.uint8)
        frame = cv2.imdecode(arr, cv2.IMREAD_COLOR)

        x = preprocess_for_model(frame)
        probs = run_inference_int8(x)

        # EMA Smoothing
        ema = EMA_ALPHA * ema + (1.0 - EMA_ALPHA) * probs
        idx = int(np.argmax(ema))
        conf = float(ema[idx])

        # Stability counter (for info only)
        if last_idx == idx:
            stable_count += 1
        else:
            stable_count = 1
            last_idx = idx

        # Logic: Update Bengali text immediately if confidence is okay
        if conf >= CONF_TH:
            word_line = BN_WORD[idx]
            bn_line = BN_MSG[idx]
            label_color = "#2ecc71" # Green
        else:
            word_line = "শনাক্ত করা হচ্ছে..."
            bn_line = "অনুগ্রহ করে হাত স্পষ্ট করে দেখান"
            label_color = "#e67e22" # Orange

        # Build UI
        top3 = np.argsort(-ema)[:3]

        content = f"""
        <div style='padding:15px; border:2px solid #ddd; border-radius:12px; width:400px; font-family:sans-serif; background-color:#f9f9f9;'>
            <div style='font-size:18px; font-weight:800; color:#333; margin-bottom:10px;'>Predicting Now: <span style='color:{label_color};'>{CLASSES[idx]}</span></div>

            <div style='background:white; padding:10px; border-radius:8px; border-left:5px solid {label_color}; margin-bottom:10px;'>
                <div style='font-size:22px; font-weight:bold; color:#2c3e50;'>{word_line}</div>
                <div style='font-size:16px; color:#7f8c8d; margin-top:4px;'>{bn_line}</div>
            </div>

            <div style='font-size:13px; color:#666;'>
                Confidence: <b>{conf*100:.1f}%</b> | Stability: {stable_count}
            </div>

            <hr style='border:0; border-top:1px solid #eee; margin:10px 0;'>
            <div style='font-size:12px; color:#999;'>Top Candidates:</div>
            {"".join([f"<div style='font-size:12px;'>{CLASSES[i]}: {ema[i]:.2f}</div>" for i in top3])}
        </div>
        """
        pred_handle.update(HTML(content))
        time.sleep(0.05)

except KeyboardInterrupt:
    print("Stopped by user.")

Running... (Interrupt to stop)
